In [1]:
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.3 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import streamlit as st
import os

RANDOM_STATE = 65

Скачать данные и обучить модель прогнозирования стоимости недвижимости (модель может быть любой сложности, даже линейная регрессия на двух признаках) - 2 балла


In [3]:
df = pd.read_csv('./realty_train_data.zip')
df.head()

,product_name,period,price,postcode,address_name,lat,lon,object_type,total_square,rooms,floor,city,settlement,district,area,description,source
0,"3-комнатная, 137 м²",NaN,63000000,127473.0,"2-й Щемиловский переулок, 5а",55.778894,37.608844,Квартира,137.0,3.0,6.0,Москва,NaN,Тверской район,NaN,Просторная квартира свободной планировки с пан...,ЦИАН
1,"Студия, 16,7 м²",NaN,3250000,108815.0,"Харлампиева, 46",55.551025,37.313054,Квартира,16.7,NaN,1.0,Москва,NaN,Филимонковское поселение,NaN,ВНИМАНИЕ! ОЧЕНЬ ПРИВЛЕКАТЕЛЬНОЕ ПРЕ...,Домклик
2,"3-комнатная, 76 м²",NaN,16004680,NaN,"ЖК Прокшино, 8 к4",55.594802,37.431264,Квартира,76.0,3.0,6.0,Москва,NaN,Сосенское поселение,NaN,"Apт.1684018. 0,01% - гибкая ипотека! Воспользу...",Яндекс.Недвижимость
3,"1-комнатная, 24 м²",NaN,7841776,NaN,"ЖК Прокшино, 6 к2",55.594332,37.428099,Квартира,24.0,1.0,10.0,Москва,NaN,Сосенское поселение,NaN,Продается однокомнатная квартира № 381 в новос...,Новострой-М
4,"3-комнатная, 126 м²",NaN,120000000,121352.0,"Давыдковская, 18",55.721097,37.464342,Квартира,126.0,3.0,16.0,Москва,NaN,Фили-Давыдково район,NaN,Шикарное предложение!\nПродаётся трёхкомнатная...,Домклик


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98822 entries, 0 to 98821
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_name  98822 non-null  object 
 1   period        0 non-null      float64
 2   price         98822 non-null  int64  
 3   postcode      93675 non-null  float64
 4   address_name  98821 non-null  object 
 5   lat           98822 non-null  float64
 6   lon           98822 non-null  float64
 7   object_type   98822 non-null  object 
 8   total_square  98822 non-null  float64
 9   rooms         94840 non-null  float64
 10  floor         98822 non-null  float64
 11  city          91928 non-null  object 
 12  settlement    6894 non-null   object 
 13  district      75111 non-null  object 
 14  area          19498 non-null  object 
 15  description   98573 non-null  object 
 16  source        98822 non-null  object 
dtypes: float64(7), int64(1), object(9)
memory usage: 12.8+ MB


In [5]:
df.describe()

,period,price,postcode,lat,lon,total_square,rooms,floor
count,0.0,9.882200e+04,93675.000000,98822.000000,98822.000000,98822.000000,94840.000000,98822.000000
mean,NaN,2.512122e+07,124503.585119,55.742691,37.586404,66.092176,2.197427,9.905274
std,NaN,3.607234e+07,11956.742109,0.107044,0.169843,48.816204,1.038628,8.219180
min,NaN,1.900000e+06,101000.000000,55.468426,37.136489,8.000000,1.000000,1.000000
25%,NaN,1.050000e+07,115516.000000,55.673101,37.471611,40.100000,1.000000,4.000000
50%,NaN,1.516713e+07,123154.000000,55.745474,37.569365,56.400000,2.000000,8.000000
75%,NaN,2.500000e+07,140003.000000,55.817697,37.689568,75.700000,3.000000,14.000000
max,NaN,1.155219e+09,143989.000000,56.028824,38.122467,2070.000000,15.000000,66.000000


In [6]:
df.isna().sum()

,0
product_name,0
period,98822
price,0
postcode,5147
address_name,1
lat,0
lon,0
object_type,0
total_square,0
rooms,3982


Удалим столбцы period, area, settlement так как в них слишком много пропусков. В остальных столбцах заполним пропуски наиболее частовстречающимся значением (после разбиения на train, test).

In [7]:
df = df.drop(columns=['period', 'area', 'settlement'])
df.head()

,product_name,price,postcode,address_name,lat,lon,object_type,total_square,rooms,floor,city,district,description,source
0,"3-комнатная, 137 м²",63000000,127473.0,"2-й Щемиловский переулок, 5а",55.778894,37.608844,Квартира,137.0,3.0,6.0,Москва,Тверской район,Просторная квартира свободной планировки с пан...,ЦИАН
1,"Студия, 16,7 м²",3250000,108815.0,"Харлампиева, 46",55.551025,37.313054,Квартира,16.7,NaN,1.0,Москва,Филимонковское поселение,ВНИМАНИЕ! ОЧЕНЬ ПРИВЛЕКАТЕЛЬНОЕ ПРЕ...,Домклик
2,"3-комнатная, 76 м²",16004680,NaN,"ЖК Прокшино, 8 к4",55.594802,37.431264,Квартира,76.0,3.0,6.0,Москва,Сосенское поселение,"Apт.1684018. 0,01% - гибкая ипотека! Воспользу...",Яндекс.Недвижимость
3,"1-комнатная, 24 м²",7841776,NaN,"ЖК Прокшино, 6 к2",55.594332,37.428099,Квартира,24.0,1.0,10.0,Москва,Сосенское поселение,Продается однокомнатная квартира № 381 в новос...,Новострой-М
4,"3-комнатная, 126 м²",120000000,121352.0,"Давыдковская, 18",55.721097,37.464342,Квартира,126.0,3.0,16.0,Москва,Фили-Давыдково район,Шикарное предложение!\nПродаётся трёхкомнатная...,Домклик


In [8]:
df.nunique()

,0
product_name,5827
price,29485
postcode,657
address_name,22083
lat,21811
lon,22060
object_type,1
total_square,5357
rooms,15
floor,61


In [9]:
df['floor'].value_counts()

,count
floor,
2.0,9076
3.0,8271
4.0,7555
5.0,7257
6.0,5663
...,...
56.0,5
54.0,4
63.0,2


In [10]:
df['city'].value_counts()

,count
city,
Москва,73180
Балашиха,2914
Химки,2330
Люберцы,1965
Красногорск,1772
Мытищи,1708
Одинцово,1247
Королёв,981
Котельники,897


In [11]:
df['source'].value_counts()

,count
source,
ЦИАН,42171
Домклик,36926
Новострой-М,10909
Яндекс.Недвижимость,8816


Удалим object_type,а та desctiption,adress_name, district.

In [12]:
df = df.drop(columns=['object_type', 'description', 'address_name', 'district'])
df.head()

,product_name,price,postcode,lat,lon,total_square,rooms,floor,city,source
0,"3-комнатная, 137 м²",63000000,127473.0,55.778894,37.608844,137.0,3.0,6.0,Москва,ЦИАН
1,"Студия, 16,7 м²",3250000,108815.0,55.551025,37.313054,16.7,NaN,1.0,Москва,Домклик
2,"3-комнатная, 76 м²",16004680,NaN,55.594802,37.431264,76.0,3.0,6.0,Москва,Яндекс.Недвижимость
3,"1-комнатная, 24 м²",7841776,NaN,55.594332,37.428099,24.0,1.0,10.0,Москва,Новострой-М
4,"3-комнатная, 126 м²",120000000,121352.0,55.721097,37.464342,126.0,3.0,16.0,Москва,Домклик


Заполним пропуски в столбце rooms и избавимся от столбца product_name. Если квартира является студией - заполняем нулем.

In [13]:
df['product_name'].str.contains('Студия').sum()

6409

In [14]:
df[df['product_name'].str.contains('Студия')]

,product_name,price,postcode,lat,lon,total_square,rooms,floor,city,source
1,"Студия, 16,7 м²",3250000,108815.0,55.551025,37.313054,16.70,NaN,1.0,Москва,Домклик
33,"Студия, 20,6 м²",7026534,109052.0,55.728307,37.739934,20.63,1.0,3.0,Москва,Новострой-М
39,"Студия, 35 м²",12925115,125438.0,55.839902,37.512305,35.00,1.0,7.0,Москва,Новострой-М
41,"Студия, 20,1 м²",7199603,109052.0,55.728307,37.739934,20.10,1.0,24.0,Москва,Новострой-М
77,"Студия, 25,8 м²",10569641,129343.0,55.848882,37.650286,25.80,1.0,30.0,Москва,Новострой-М
...,...,...,...,...,...,...,...,...,...,...
98762,"Студия, 100,1 м²",77500000,125047.0,55.770896,37.591538,100.10,NaN,10.0,Москва,Домклик
98770,"Студия, 23,5 м²",7220895,143441.0,55.860761,37.377363,23.46,1.0,24.0,Москва,Новострой-М
98791,"Студия, 170 м²",95000000,127051.0,55.772778,37.627077,170.00,NaN,4.0,Москва,Домклик
98795,"Студия, 30 м²",6100000,140016.0,55.694806,37.955912,30.00,NaN,17.0,Люберцы,Домклик


Видно, что в некоторых строках студии соответствует количество комнат - 1. Для однокомнатных квартир это значение так же 1. Будем ставить в соответствие студии 0.

In [15]:
df.loc[df['product_name'].str.contains('Студия'), 'rooms'] = 0
df['rooms'].value_counts()

,count
rooms,
2.0,34278
3.0,24885
1.0,23859
0.0,6409
4.0,6379
5.0,1618
6.0,649
7.0,45
8.0,23


In [16]:
df.isna().sum()

,0
product_name,0
price,0
postcode,5147
lat,0
lon,0
total_square,0
rooms,652
floor,0
city,6894
source,0


Осталось некоторое количество строк с незаполненным значением. Посмотрим на них.

In [17]:
df[df['rooms'].isna()]

,product_name,price,postcode,lat,lon,total_square,rooms,floor,city,source
52,"Квартира, 40,5 м²",30142700,107078.0,55.771159,37.643736,40.46,NaN,9.0,Москва,ЦИАН
520,"Квартира, 170 м²",85000000,125047.0,55.770896,37.591538,170.00,NaN,7.0,Москва,ЦИАН
531,"Квартира, 31,1 м²",11687380,117420.0,55.662007,37.553279,31.10,NaN,24.0,Москва,ЦИАН
555,"Квартира, 30,7 м²",13808860,129343.0,55.848882,37.650286,30.70,NaN,20.0,Москва,ЦИАН
567,"Квартира, 57 м²",13800000,125424.0,55.825023,37.415519,57.00,NaN,2.0,Москва,ЦИАН
...,...,...,...,...,...,...,...,...,...,...
98421,"Квартира, 103 м²",70700000,119019.0,55.749802,37.596572,103.00,NaN,5.0,Москва,ЦИАН
98430,"Квартира, 28,1 м²",19999999,119192.0,55.695901,37.485901,28.10,NaN,9.0,Москва,ЦИАН
98439,"Квартира, 129,6 м²",62538652,129085.0,55.807549,37.635721,129.55,NaN,15.0,Москва,ЦИАН
98673,"Квартира, 26,1 м²",12379230,129343.0,55.848882,37.650286,26.10,NaN,31.0,Москва,ЦИАН


Разобьем датасет на тестовую и обучающую выборки.

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=RANDOM_STATE)

Заполним пропуски (postcode, rooms, city, district) в тестовой и обучающей выборках отдельно


Для столбца postcode заполним случайными значениями из 10 наиболее часто встречаемых индексов.

In [20]:
most_freq_postcodes = X_train['postcode'].value_counts().index[:10]

In [21]:
np.random.randint(0,10,10)

array([0, 4, 4, 4, 7, 1, 5, 9, 6, 3])

In [22]:
most_freq_postcodes[np.random.randint(0,10,15)]

Index([127287.0, 127427.0, 109316.0, 127287.0, 123423.0, 143441.0, 119192.0,
       123290.0, 123423.0, 127287.0, 123423.0, 119192.0, 108814.0, 123290.0,
       123423.0],
      dtype='float64', name='postcode')

In [23]:
X_train['postcode'].isna().sum()

3478

In [24]:
X_train.loc[X_train['postcode'].isna(), 'postcode'] = most_freq_postcodes[np.random.randint(0,10,X_train['postcode'].isna().sum())]

Заполняем rooms

In [25]:
X_train.loc[X_train['rooms'].isna(), 'rooms'] = X_train['rooms'].value_counts().index[0]

Заполняем city

In [26]:
X_train.loc[X_train['city'].isna(), 'city'] = X_train['city'].value_counts().index[0]

Заполняем district

Теперь заполним пропуски в тестовой выборке

In [27]:
most_freq_postcodes_test = X_test['postcode'].value_counts().index[:10]

In [28]:
X_test.loc[X_test['postcode'].isna(), 'postcode'] = most_freq_postcodes_test[np.random.randint(0,10,X_test['postcode'].isna().sum())]

In [29]:
X_test.loc[X_test['rooms'].isna(), 'rooms'] = X_test['rooms'].value_counts().index[0]

In [30]:
X_test.loc[X_test['city'].isna(), 'city'] = X_test['city'].value_counts().index[0]

In [31]:
X_train.isna().sum()

,0
product_name,0
postcode,0
lat,0
lon,0
total_square,0
rooms,0
floor,0
city,0
source,0


In [32]:
X_test.isna().sum()

,0
product_name,0
postcode,0
lat,0
lon,0
total_square,0
rooms,0
floor,0
city,0
source,0


Поле product_name содержит информацию, которая есть в других столбцах, поэтому от него можно избавиться.

In [33]:
X_train = X_train.drop(columns=['product_name'])

In [34]:
X_test = X_test.drop(columns=['product_name'])

Закодируем категориальные признаки

In [35]:
df_cols = ['city', 'source']
df_1 = X_train.drop(columns = df_cols, axis = 1)
df_2 = pd.get_dummies(X_train[df_cols])

X_train = pd.concat([df_1, df_2], axis=1, join='inner')

In [36]:
df_cols = ['city', 'source']
df_1 = X_test.drop(columns = df_cols, axis = 1)
df_2 = pd.get_dummies(X_test[df_cols])

X_test = pd.concat([df_1, df_2], axis=1, join='inner')

In [42]:
X_train.head()

,postcode,lat,lon,total_square,rooms,floor,city_Балашиха,city_Видное,city_Дзержинский,city_Долгопрудный,...,city_Подольск,city_Пушкино,city_Реутов,city_Химки,city_Щербинка,city_Щёлково,source_Домклик,source_Новострой-М,source_ЦИАН,source_Яндекс.Недвижимость
85752,143005.0,55.685334,37.308585,66.2,3.0,1.0,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
37739,111675.0,55.714497,37.891466,42.0,1.0,10.0,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
36384,115487.0,55.671773,37.659661,30.0,1.0,1.0,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
21236,143909.0,55.836367,37.939348,36.0,1.0,22.0,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False
84047,125167.0,55.792538,37.543053,113.2,3.0,2.0,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False


Обучим регрессионную модель, основанную на случайных деревьях.

In [37]:
from sklearn.ensemble import RandomForestRegressor

rg = RandomForestRegressor(n_estimators=10, random_state=RANDOM_STATE)
rg.fit(X_train, y_train)

RandomForestRegressor(n_estimators=10, random_state=65)

In [38]:
import pickle

with open('rf_fitted.pkl', 'wb') as file:
  pickle.dump(rg, file)

**2. Реализуйте код для получения предсказания обученной моделью - 1 балл**

In [50]:
def rg_predict(X_in, model_path):
  with open(model_path, 'rb') as file:
    model = pickle.load(file)

    df_cols = ['city', 'source']
    df_1 = X_in.drop(columns = df_cols, axis = 1)

    # Список всех возможных городов и источников
    all_cities = ['Балашиха',
      'Видное',
      'Дзержинский',
      'Долгопрудный',
      'Ивантеевка',
      'Королёв',
      'Котельники',
      'Красногорск',
      'Лобня',
      'Лыткарино',
      'Люберцы',
      'Москва',
      'Московский',
      'Мытищи',
      'Одинцово',
      'Подольск',
      'Пушкино',
      'Реутов',
      'Химки',
      'Щербинка',
      'Щёлково']

    all_sources = ["Домклик", "Новострой-М", "ЦИАН", "Яндекс.Недвижимость"]

    # Удаляем категориальные признаки
    df_cols = ['city', 'source']
    df_1 = X_in.drop(columns=df_cols, axis=1)

    # Создаем фиктивные переменные с учетом всех возможных категорий
    df_2 = pd.get_dummies(X_in[df_cols], columns=['city', 'source'])
    df_2 = df_2.reindex(columns=[f'city_{city}' for city in all_cities] + [f'source_{source}' for source in all_sources], fill_value=0)

    X_in = pd.concat([df_1, df_2], axis=1, join='inner')

    preds = model.predict(X_in)
  return preds

Часть кода для интерфейса находится во втором файле.